In [ ]:
import h5py
import torch
import torchvision.transforms.v2 as transforms_v2

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
inv_apsdvs_max = 1 / 255
def normalise_dvsaps(data):
    # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
    return data * inv_apsdvs_max

transform_list = [
    lambda x: normalise_dvsaps(x),
    transforms_v2.ToImage(),
    transforms_v2.ToDtype(torch.float32, scale=False),
    transforms_v2.Normalize(
        # aps mean, dvs mean
        mean=[0.4827, 0.3458],
        std=[0.4672, 0.1243]
    )
]
transform = transforms_v2.Compose(transform_list)

# pick the 40th frame
frame_index = 40
with h5py.File("test.hdf5", "r") as f:
    data_aps = f["aps_frame_224x224"]
    data_dvs = f["dvs_frame_224x224"]
    dvs_frame = data_dvs[frame_index]
    aps_frame = data_aps[frame_index]

stacked_frame = np.stack((aps_frame, dvs_frame), axis=-1) # h, w, c
transformed_frame = transform(stacked_frame) # c, h, w
transformed_aps_np = transformed_frame[0].numpy()
transformed_dvs_np = transformed_frame[1].numpy()

print(f"aps mean: {np.mean(transformed_aps_np)} aps stddev: {np.std(transformed_aps_np)}")
print(f"old aps mean (divided): {np.mean(aps_frame) / 255} old aps stddev (divided): {np.std(aps_frame) / 255}")
print(f"dvs mean: {np.mean(transformed_dvs_np)} dvs stddev: {np.std(transformed_dvs_np)}")
print(f"old dvs mean (divided): {np.mean(dvs_frame) / 255} old dvs stddev (divided): {np.std(dvs_frame) / 255}")

# Rescale normalized values to [0, 1] for better visualization
# aps_for_vis = (transformed_aps_np - transformed_aps_np.min()) / (transformed_aps_np.max() - transformed_aps_np.min())
# dvs_for_vis = (transformed_dvs_np - transformed_dvs_np.min()) / (transformed_dvs_np.max() - transformed_dvs_np.min())
aps_for_vis = transformed_aps_np
dvs_for_vis = transformed_dvs_np
# Create an RGB image
rgb_image = np.zeros((aps_for_vis.shape[0], aps_for_vis.shape[1], 3))
rgb_image[:, :, 0] = aps_for_vis  # Red channel - APS
rgb_image[:, :, 1] = dvs_for_vis  # Green channel - DVS
# Blue channel remains zeros
# Display individual channels and combined image
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
# Original APS
axes[0].imshow(transformed_aps_np, origin="lower", cmap='Reds')
axes[0].set_title("APS Channel")
# Original DVS
axes[1].imshow(transformed_dvs_np, origin="lower", cmap='Greens')
axes[1].set_title("DVS Channel")
# Combined RGB
axes[2].imshow(rgb_image, origin="lower")
axes[2].set_title("APS (red) and DVS (green) combined")
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image

# Load the RGB image
image = Image.open("road.jpg")
rgb_image = np.array(image)
print(f"Image shape: {rgb_image.shape}")  # Should be (H, W, 3)

# Create transforms for RGB image
transform_list = [
    transforms_v2.ToImage(),
    transforms_v2.ToDtype(torch.float32, scale=True),  # scale=True divides by 255
    transforms_v2.Normalize(
        # ImageNet mean and std values
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
]
transform = transforms_v2.Compose(transform_list)

# Apply transforms
transformed_image = transform(rgb_image)  # Output shape: (3, H, W)
print(f"Transformed image shape: {transformed_image.shape}")

# Convert to numpy for visualization
r_channel = transformed_image[0].numpy()
g_channel = transformed_image[1].numpy()
b_channel = transformed_image[2].numpy()

# Print statistics
print(f"Red channel - mean: {np.mean(r_channel):.4f}, stddev: {np.std(r_channel):.4f}")
print(f"Green channel - mean: {np.mean(g_channel):.4f}, stddev: {np.std(g_channel):.4f}")
print(f"Blue channel - mean: {np.mean(b_channel):.4f}, stddev: {np.std(b_channel):.4f}")

# Rescale normalized values to [0, 1] for better visualization
r_for_vis = (r_channel - r_channel.min()) / (r_channel.max() - r_channel.min())
g_for_vis = (g_channel - g_channel.min()) / (g_channel.max() - g_channel.min())
b_for_vis = (b_channel - b_channel.min()) / (b_channel.max() - b_channel.min())

# Visualize individual channels and combined image
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Individual channels
axes[0].imshow(r_for_vis, cmap='Reds')
axes[0].set_title("Red Channel")

axes[1].imshow(g_for_vis, cmap='Greens')
axes[1].set_title("Green Channel")

axes[2].imshow(b_for_vis, cmap='Blues')
axes[2].set_title("Blue Channel")

# Original RGB image
axes[3].imshow(rgb_image)
axes[3].set_title("Original RGB Image")

plt.tight_layout()
plt.show()

In [ ]:
def get_model_structure(model, prefix=None):
    structure = {}
    for name, param in model.named_parameters():
        if prefix:
            name = name[len(prefix) + 1:]
        structure[name] = param.shape
    return structure

import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from models.video_transformers.modified_swin3d_t import ModifiedSwin3Dt


model_dvsaps = ModifiedSwin3Dt(in_channels=2)
model_rgb = ModifiedSwin3Dt(in_channels=3)

from torchvision.models.video import swin3d_t

model_rgb_original = swin3d_t(num_classes=1)

dvsaps_structure = get_model_structure(model_dvsaps, prefix="model")
original_structure = get_model_structure(model_rgb_original)

# Find differences
dvsaps_only = {k: v for k, v in dvsaps_structure.items() if k not in original_structure}
original_only = {k: v for k, v in original_structure.items() if k not in dvsaps_structure}
common_keys = set(dvsaps_structure.keys()) & set(original_structure.keys())
different_shapes = {k: (dvsaps_structure[k], original_structure[k]) 
                   for k in common_keys if dvsaps_structure[k] != original_structure[k]}

print(f"Parameters only in your modified model: {len(dvsaps_only)}")
print(f"Parameters only in original model: {len(original_only)}")
print(f"Parameters with different shapes: {len(different_shapes)}")

# Display some examples of differences
if dvsaps_only:
    print("\nSample parameters only in your model:")
    for k, v in list(dvsaps_only.items())[:5]:
        print(f"  {k}: {v}")

if original_only:
    print("\nSample parameters only in original model:")
    for k, v in list(original_only.items())[:5]:
        print(f"  {k}: {v}")
        
if different_shapes:
    print("\nSample parameters with different shapes:")
    for k, v in list(different_shapes.items())[:5]:
        print(f"  {k}: {v[0]} vs {v[1]}")
